# UNIVERSIDAD EAN
# Proyecto integrador — Ciencia de datos
## Análisis de amenazas de ciberseguridad
**Dataset:** UNSW-NB15  
**Pregunta central:** ¿Cómo podemos usar la ciencia de datos para identificar patrones asociados a ataques informáticos y anticipar posibles amenazas?

---
**Parte 1** Dataset → **Parte 2** Estadística y gráficas → **Parte 3** Modelo predictivo

> **Colab:** Archivo → Subir notebook → Entorno de ejecución → **Reiniciar y ejecutar todo**.

# PARTE 1 — Dataset

## 1.1 Justificación del dataset

Elegimos el dataset **UNSW-NB15** porque reúne tráfico de red real mezclado con ataques modernos (DoS, Exploits, Reconnaissance, etc.).  
Nos permite estudiar, con datos concretos, la diferencia entre actividad **normal** y **maliciosa**.  
Se relaciona con la carrera porque une **ciencia de datos** y **ciberseguridad**: convertir registros de red en información útil para detectar amenazas.  
Además, ya viene etiquetado (`label` y `attack_cat`), lo que facilita entrenar y evaluar un modelo predictivo.  
Fuente oficial: [UNSW Canberra — UNSW-NB15](https://research.unsw.edu.au/projects/unsw-nb15-dataset).

## 1.2 Importación de librerías y carpetas

**Qué hacemos:** cargamos las librerías del proyecto y creamos carpetas de salida.  
**Por qué:** el notebook debe correr completo y guardar CSV y gráficas de forma ordenada.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

os.makedirs("data", exist_ok=True)
os.makedirs("img", exist_ok=True)

print("=" * 65)
print("PROYECTO DE CIENCIA DE DATOS - UNIVERSIDAD EAN")
print("ANÁLISIS DE AMENAZAS DE CIBERSEGURIDAD | UNSW-NB15")
print("=" * 65)
print("Carpetas listas: data/ e img/")

## 1.3 Carga de datos

**Qué hacemos:** descargamos el training set de UNSW-NB15.  
**Por qué:** necesitamos los datos en memoria para explorar, limpiar y modelar.

**Si falla la URL:** usa la celda de subida manual.

In [ ]:
URL_DATOS = (
    "https://raw.githubusercontent.com/stijnhuysman/DissertationData/main/"
    "UNSW_NB15_training-set.csv"
)

print("Descargando dataset... (puede tardar 20-60 segundos)")
df = pd.read_csv(URL_DATOS)

print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print("\nPrimeras 5 filas:")
display(df.head())
print("\nNombres de columnas:")
print(list(df.columns))

### Alternativa: subir el CSV a mano

In [ ]:
# --- SOLO SI FALLÓ LA DESCARGA ---
# from google.colab import files
# uploaded = files.upload()
# nombre = list(uploaded.keys())[0]
# df = pd.read_csv(nombre)
# print(df.shape)
# display(df.head())

print("Celda de respaldo lista. No hace falta si la descarga funcionó.")

## 1.4 Documentación de variables

**Qué hacemos:** tabla con cada columna, su tipo y su significado.  
**Por qué:** el informe pide documentar fuente, variables y tipos.

In [ ]:
significado = {
    "id": "Identificador de la fila",
    "dur": "Duración de la conexión (segundos)",
    "proto": "Protocolo de red (tcp, udp, icmp...)",
    "service": "Servicio detectado (http, dns, ftp...) o '-' si no aplica",
    "state": "Estado de la conexión (FIN, INT, CON...)",
    "spkts": "Paquetes enviados por el origen",
    "dpkts": "Paquetes enviados por el destino",
    "sbytes": "Bytes enviados por el origen",
    "dbytes": "Bytes enviados por el destino",
    "rate": "Tasa de paquetes por segundo",
    "sttl": "TTL (tiempo de vida) en paquetes del origen",
    "dttl": "TTL en paquetes del destino",
    "sload": "Carga de bits/s desde el origen",
    "dload": "Carga de bits/s desde el destino",
    "sloss": "Paquetes perdidos del origen",
    "dloss": "Paquetes perdidos del destino",
    "sinpkt": "Intervalo promedio entre paquetes del origen",
    "dinpkt": "Intervalo promedio entre paquetes del destino",
    "sjit": "Jitter (variación de retardo) del origen",
    "djit": "Jitter del destino",
    "swin": "Ventana TCP del origen",
    "stcpb": "Número de secuencia TCP del origen",
    "dtcpb": "Número de secuencia TCP del destino",
    "dwin": "Ventana TCP del destino",
    "tcprtt": "Tiempo de ida y vuelta TCP (RTT)",
    "synack": "Tiempo entre SYN y SYN-ACK",
    "ackdat": "Tiempo entre SYN-ACK y ACK",
    "smean": "Tamaño medio de paquetes del origen",
    "dmean": "Tamaño medio de paquetes del destino",
    "trans_depth": "Profundidad de transacciones HTTP",
    "response_body_len": "Tamaño del cuerpo de respuesta HTTP",
    "ct_srv_src": "Conexiones al mismo servicio desde la misma IP origen",
    "ct_state_ttl": "Conteo según estado y TTL",
    "ct_dst_ltm": "Conexiones a la misma IP destino en ventana corta",
    "ct_src_dport_ltm": "Conexiones origen–puerto destino en ventana corta",
    "ct_dst_sport_ltm": "Conexiones destino–puerto origen en ventana corta",
    "ct_dst_src_ltm": "Conexiones entre el mismo par origen-destino",
    "is_ftp_login": "1 si hubo login FTP, 0 si no",
    "ct_ftp_cmd": "Comandos FTP en el flujo",
    "ct_flw_http_mthd": "Métodos HTTP en el flujo",
    "ct_src_ltm": "Conexiones desde la misma IP origen en ventana corta",
    "ct_srv_dst": "Conexiones al mismo servicio hacia la misma IP destino",
    "is_sm_ips_ports": "1 si origen y destino usan misma IP/puerto",
    "attack_cat": "Categoría del ataque (o Normal)",
    "label": "Variable objetivo: 0 = normal, 1 = ataque",
}

doc = pd.DataFrame({
    "variable": df.columns,
    "tipo_pandas": [str(df[c].dtype) for c in df.columns],
    "significado": [significado.get(c, "(revisar documentación oficial)") for c in df.columns],
})

print("Fuente: UNSW Canberra Cyber Range Lab (UNSW-NB15 training set)")
print(f"Registros: {len(df):,} | Variables: {df.shape[1]}")
display(doc)

print("\nCómo interpretarlo:")
print("- 'object' = texto/categoría; 'int64'/'float64' = números.")
print("- 'label' es la etiqueta que usaremos para predecir ataque vs normal.")

## 1.5 Exploración inicial

**Qué hacemos:** revisamos tipos, nulos, duplicados y distribución normal/ataque.  
**Por qué:** antes de limpiar hay que ver qué problemas hay.

In [ ]:
print("=== Tipos de datos ===")
display(df.dtypes.value_counts().rename("cantidad"))

print("\n=== Valores nulos (solo si hay) ===")
nulos = df.isna().sum()
nulos = nulos[nulos > 0]
print("No hay valores nulos (NaN)." if len(nulos) == 0 else nulos.sort_values(ascending=False))

print("\n=== Duplicados ===")
print(f"Filas duplicadas exactas: {df.duplicated().sum():,}")

print("\n=== Distribución de label ===")
display(df["label"].value_counts().rename(index={0: "Normal (0)", 1: "Ataque (1)"}))
display((df["label"].value_counts(normalize=True) * 100).round(2).rename(
    index={0: "Normal %", 1: "Ataque %"}
))

print("\n=== Categorías de ataque ===")
display(df["attack_cat"].value_counts())

print("\n=== service (el '-' = sin servicio) ===")
display(df["service"].value_counts().head(10))

cols_clave = [c for c in ["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sload", "dload"] if c in df.columns]
print("\n=== Resumen numérico de variables clave ===")
display(df[cols_clave].describe().T)

print("\nCómo interpretarlo:")
print("- Si una clase domina (más ataques o más normal), el modelo puede sesgarse.")
print("- El '-' en service no es error: significa 'sin servicio identificado'.")
print("- Valores muy altos en sbytes/rate pueden ser tráfico intenso o indicios de ataque.")

## 1.6 Limpieza de datos

**Qué hacemos:** quitamos columnas inútiles, unificamos textos, tratamos `-` en service y eliminamos duplicados.  
**Por qué:** datos limpios = estadísticas y modelos más confiables.

In [ ]:
filas_antes, cols_antes = df.shape
df_clean = df.copy()

if "id" in df_clean.columns:
    df_clean = df_clean.drop(columns=["id"])
    print("Decisión: eliminamos 'id' (no describe el comportamiento del tráfico).")

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
print("Decisión: quitamos espacios en textos.")

if "service" in df_clean.columns:
    df_clean["service"] = df_clean["service"].replace({"-": "none", "nan": "none"})
    print("Decisión: '-' en service → 'none'.")

for col in ["proto", "state", "service", "attack_cat"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.lower()
print("Decisión: categorías a minúsculas.")

df_clean["label"] = pd.to_numeric(df_clean["label"], errors="coerce")
n_label_malo = df_clean["label"].isna().sum()
df_clean = df_clean.dropna(subset=["label"])
df_clean["label"] = df_clean["label"].astype(int)
print(f"Decisión: label como entero 0/1. Filas inválidas eliminadas: {n_label_malo}")

if "attack_cat" in df_clean.columns:
    incoherentes = (
        ((df_clean["label"] == 0) & (df_clean["attack_cat"] != "normal"))
        | ((df_clean["label"] == 1) & (df_clean["attack_cat"] == "normal"))
    ).sum()
    print(f"Filas con posible incoherencia label/attack_cat: {incoherentes}")

dup_antes = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Decisión: eliminamos {dup_antes:,} duplicados exactos.")

nulos_restantes = df_clean.isna().sum().sum()
if nulos_restantes > 0:
    for col in df_clean.columns:
        if df_clean[col].isna().any():
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            else:
                moda = df_clean[col].mode()
                df_clean[col] = df_clean[col].fillna(moda.iloc[0] if len(moda) else "desconocido")
    print(f"Decisión: rellenamos {nulos_restantes} nulos (mediana/moda).")
else:
    print("No quedaron nulos que rellenar.")

filas_despues, cols_despues = df_clean.shape
print(f"\nAntes: {filas_antes:,} x {cols_antes} | Después: {filas_despues:,} x {cols_despues}")

## 1.7 Resumen depurado y guardado del CSV

**Qué hacemos:** comparamos antes/después y guardamos el CSV limpio.  
**Por qué:** deja evidencia de la depuración y un archivo listo para Partes 2 y 3.

In [ ]:
resumen = pd.DataFrame({
    "métrica": ["Filas", "Columnas", "Nulos totales", "Duplicados", "% Normal", "% Ataque"],
    "antes": [
        filas_antes,
        cols_antes,
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        round((df["label"] == 0).mean() * 100, 2),
        round((df["label"] == 1).mean() * 100, 2),
    ],
    "después": [
        filas_despues,
        cols_despues,
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        round((df_clean["label"] == 0).mean() * 100, 2),
        round((df_clean["label"] == 1).mean() * 100, 2),
    ],
})

print("=== Comparación antes vs después ===")
display(resumen)
display(df_clean.head())

ruta_csv = "data/UNSW_NB15_limpio.csv"
df_clean.to_csv(ruta_csv, index=False)
print(f"\nCSV limpio guardado en: {ruta_csv}")
# from google.colab import files
# files.download(ruta_csv)

print("\nCómo interpretarlo:")
print("- Si bajaron filas: duplicados o etiquetas inválidas.")
print("- Si bajó una columna: normalmente fue 'id'.")
print("- data/UNSW_NB15_limpio.csv es la base para estadística y el modelo.")

### Texto para el video — Parte 1

En esta primera parte elegimos UNSW-NB15 porque muestra tráfico normal y ataques etiquetados. Documentamos cada variable, exploramos nulos, duplicados y la proporción entre actividad normal y maliciosa. Luego limpiamos textos, unificamos el servicio “sin identificar”, quitamos duplicados y la columna id, y guardamos un CSV depurado. Con eso ya tenemos una base confiable para estadísticas y para entrenar un modelo que ayude a anticipar amenazas.

# PARTE 2 — Estadística descriptiva y gráficas

## 2.1 Media, mediana y moda

**Qué hacemos:** calculamos media (promedio), mediana (valor del medio) y moda (valor más frecuente) en variables clave.  
**Por qué:** resumen cómo se comporta el tráfico y si hay valores extremos que “jalan” el promedio.

In [ ]:
# Variables numéricas clave para el análisis
vars_clave = [c for c in ["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sload", "dload", "sttl", "smean", "dmean"] if c in df_clean.columns]

filas_stats = []
for col in vars_clave:
    serie = df_clean[col]
    moda = serie.mode()
    filas_stats.append({
        "variable": col,
        "media": round(serie.mean(), 4),
        "mediana": round(serie.median(), 4),
        "moda": round(moda.iloc[0], 4) if len(moda) else np.nan,
    })

tabla_mmm = pd.DataFrame(filas_stats)
display(tabla_mmm)

print("\nCómo interpretarlo:")
print("- Si media >> mediana, hay valores muy altos (colas largas): típico en bytes o rate.")
print("- La moda muestra el valor más común; útil en variables como sttl.")
print("- En ciberseguridad, promedios altos de sbytes/rate pueden asociarse a tráficos intensos o ataques.")

## 2.2 Rango, varianza y desviación estándar

**Qué hacemos:** medimos dispersión: rango (máx−mín), varianza y desviación estándar (qué tan lejos están los datos del promedio).  
**Por qué:** mucha dispersión sugiere comportamientos muy distintos entre conexiones (normal vs ataque).

In [ ]:
filas_disp = []
for col in vars_clave:
    serie = df_clean[col]
    filas_disp.append({
        "variable": col,
        "mín": round(serie.min(), 4),
        "máx": round(serie.max(), 4),
        "rango": round(serie.max() - serie.min(), 4),
        "varianza": round(serie.var(), 4),
        "desv_estándar": round(serie.std(), 4),
    })

tabla_disp = pd.DataFrame(filas_disp)
display(tabla_disp)

print("\nCómo interpretarlo:")
print("- Rango grande = hay conexiones muy distintas (casi vacías vs muy pesadas).")
print("- Desviación alta = los valores se alejan mucho del promedio.")
print("- En seguridad, alta dispersión en rate/sbytes puede indicar picos anómalos dignos de alerta.")

## 2.3 Comparación: actividad normal vs maliciosa

**Qué hacemos:** comparamos promedios de variables clave entre `label=0` (normal) y `label=1` (ataque).  
**Por qué:** buscar diferencias que ayuden a distinguir amenazas.

In [ ]:
df_clean["tipo_trafico"] = df_clean["label"].map({0: "Normal", 1: "Malicioso"})

comparacion = (
    df_clean.groupby("tipo_trafico")[vars_clave]
    .mean()
    .T
    .rename(columns={"Normal": "media_normal", "Malicioso": "media_malicioso"})
)
comparacion["diferencia"] = comparacion["media_malicioso"] - comparacion["media_normal"]
display(comparacion.round(4))

print("\nCómo interpretarlo:")
print("- Si media_malicioso es claramente mayor/menor, esa variable ayuda a separar clases.")
print("- Ejemplo típico: sttl o rate suelen comportarse distinto en ataques.")
print("- Estas diferencias alimentan las hipótesis y el modelo predictivo.")

## 2.4 Gráfica de barras: normal vs maliciosa

**Qué hacemos:** contamos cuántos registros hay de cada tipo.  
**Por qué:** ver el balance de clases de un vistazo.

In [ ]:
conteo = df_clean["tipo_trafico"].value_counts().reindex(["Normal", "Malicioso"])

fig, ax = plt.subplots()
barras = ax.bar(conteo.index, conteo.values, color=["#2E7D32", "#C62828"])
ax.set_title("Actividad normal vs maliciosa")
ax.set_xlabel("Tipo de tráfico")
ax.set_ylabel("Cantidad de registros")
for b, v in zip(barras, conteo.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("img/barras_normal_vs_maliciosa.png", dpi=150, bbox_inches="tight")
plt.show()

print("Cómo interpretarlo:")
print("- La barra más alta es la clase mayoritaria.")
print("- Un desbalance fuerte afecta accuracy: por eso también miramos recall de ataque.")

## 2.5 Histogramas de variables clave

**Qué hacemos:** mostramos la distribución de algunas variables (separando normal y malicioso).  
**Por qué:** ver forma de los datos y posibles diferencias entre clases.

In [ ]:
vars_hist = [c for c in ["dur", "rate", "sbytes", "sttl"] if c in df_clean.columns]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, vars_hist):
    # Usamos log1p para variables muy sesgadas (valores extremos)
    for nombre, color in [("Normal", "#2E7D32"), ("Malicioso", "#C62828")]:
        datos = np.log1p(df_clean.loc[df_clean["tipo_trafico"] == nombre, col].clip(lower=0))
        ax.hist(datos, bins=40, alpha=0.55, label=nombre, color=color)
    ax.set_title(f"Histograma de {col} (escala log1p)")
    ax.set_xlabel(f"log1p({col})")
    ax.set_ylabel("Frecuencia")
    ax.legend()

plt.tight_layout()
plt.savefig("img/histogramas_variables_clave.png", dpi=150, bbox_inches="tight")
plt.show()

print("Cómo interpretarlo:")
print("- Usamos log1p porque bytes/rate tienen colas muy largas; así se ve mejor la forma.")
print("- Si las curvas verde y roja se separan, esa variable discrimina bien.")

## 2.6 Outliers (método IQR) y boxplots

**Qué hacemos:** detectamos valores atípicos con el rango intercuartílico (IQR) y los visualizamos con boxplots.  
**Por qué:** en seguridad, un outlier puede ser un pico de ataque… o un error de medición.

In [ ]:
def contar_outliers_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (serie < lim_inf) | (serie > lim_sup)
    return int(mask.sum()), lim_inf, lim_sup

vars_out = [c for c in ["dur", "rate", "sbytes", "dbytes", "sload"] if c in df_clean.columns]
res_out = []
for col in vars_out:
    n_out, lo, hi = contar_outliers_iqr(df_clean[col])
    res_out.append({
        "variable": col,
        "outliers": n_out,
        "%_outliers": round(100 * n_out / len(df_clean), 2),
        "limite_inf": round(lo, 4),
        "limite_sup": round(hi, 4),
    })

display(pd.DataFrame(res_out))

fig, axes = plt.subplots(1, len(vars_out), figsize=(3.2 * len(vars_out), 5))
if len(vars_out) == 1:
    axes = [axes]

for ax, col in zip(axes, vars_out):
    sns.boxplot(
        data=df_clean,
        x="tipo_trafico",
        y=col,
        ax=ax,
        palette={"Normal": "#2E7D32", "Malicioso": "#C62828"},
        showfliers=True,
    )
    ax.set_title(f"Boxplot: {col}")
    ax.set_xlabel("Tipo de tráfico")
    ax.set_ylabel(col)
    # Escala log para leer mejor valores extremos
    if df_clean[col].max() > 0:
        ax.set_yscale("symlog")

plt.tight_layout()
plt.savefig("img/boxplots_outliers.png", dpi=150, bbox_inches="tight")
plt.show()

print("Cómo interpretarlo:")
print("- IQR: valores fuera de [Q1-1.5·IQR, Q3+1.5·IQR] se consideran atípicos.")
print("- En ciberseguridad, outliers de rate/sbytes pueden ser escaneos, DoS o transferencias anómalas.")
print("- No los borramos automáticamente: a menudo son la señal del ataque.")

## 2.7 Mapa de calor de correlaciones

**Qué hacemos:** medimos qué tan relacionadas están las variables numéricas entre sí (y con `label`).  
**Por qué:** variables muy correlacionadas con el ataque son candidatas fuertes para el modelo.

In [ ]:
# Tomamos variables numéricas + label (sin saturar el gráfico)
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
# Priorizamos variables clave + label
prefs = [c for c in vars_clave + ["ct_state_ttl", "ct_dst_src_ltm", "ct_srv_dst", "label"] if c in num_cols]
corr = df_clean[prefs].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=False, cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Mapa de calor de correlaciones (variables clave)")
plt.tight_layout()
plt.savefig("img/mapa_calor_correlaciones.png", dpi=150, bbox_inches="tight")
plt.show()

print("Correlación de cada variable con label (ataque):")
corr_label = corr["label"].drop("label").sort_values(key=abs, ascending=False)
display(corr_label.round(4))

print("\nCómo interpretarlo:")
print("- Correlación cerca de +1 o -1: relación fuerte (positiva o inversa).")
print("- Cerca de 0: poca relación lineal.")
print("- Las más correlacionadas con label suelen ser las más útiles para predecir ataques.")

## 2.8 Hipótesis o relaciones encontradas

**Qué hacemos:** planteamos 2–3 hipótesis a partir de lo observado.  
**Por qué:** conecta la exploración con el modelo y con el relato del video.

In [ ]:
print("Hipótesis basadas en la exploración:")
print()
print("H1. El tráfico malicioso presenta valores de sttl / rate distintos al tráfico normal,")
print("    por lo que esas variables ayudan a separar clases.")
print()
print("H2. Variables de volumen (sbytes, dbytes, sload) tienen muchos outliers;")
print("    en seguridad esos picos pueden asociarse a ataques de alto volumen (p. ej. DoS).")
print()
print("H3. Conteos de conexiones recientes (ct_*) se relacionan con label:")
print("    patrones de muchas conexiones en poco tiempo pueden indicar reconocimiento o abuso.")
print()
print("Cómo interpretarlo:")
print("- Son hipótesis de trabajo: el modelo de la Parte 3 las pondrá a prueba.")
print("- Si el recall de ataque es alto y esas variables salen importantes, se refuerzan.")

### Texto para el video — Parte 2

En la Parte 2 calculamos media, mediana y moda, y vimos que varias variables tienen colas largas. Medimos dispersión y comparamos tráfico normal contra malicioso: hay diferencias claras en indicadores como rate o sttl. Detectamos outliers con IQR y boxplots; en ciberseguridad esos extremos a menudo son la señal, no el ruido. Con el mapa de calor identificamos variables más ligadas al ataque y planteamos hipótesis que el modelo va a evaluar.

# PARTE 3 — Modelo predictivo

## 3.1 Variable objetivo y preparación de datos

**Qué hacemos:** definimos `label` (0=normal, 1=ataque), preparamos features numéricas y separamos entrenamiento/prueba.  
**Por qué:** el modelo aprende patrones en train y se evalúa en test (datos que no vio).

In [ ]:
# Variable objetivo
y = df_clean["label"].astype(int)

# Features: numéricas, sin filtrar label ni columnas de fuga de información
# attack_cat describe el tipo de ataque: no se usa como entrada (sería "hacer trampa")
cols_excluir = {"label", "attack_cat", "tipo_trafico"}
X = df_clean.drop(columns=[c for c in cols_excluir if c in df_clean.columns])
X = X.select_dtypes(include=[np.number]).copy()

print("Variable objetivo: label → 0 = Normal, 1 = Ataque")
print(f"Features numéricas usadas: {X.shape[1]}")
print(list(X.columns))

# Separación 80% train / 20% test (estratificada para mantener proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print("Proporción de ataques en train:", round(y_train.mean(), 4))
print("Proporción de ataques en test:", round(y_test.mean(), 4))

# Escalado: ayuda a la regresión logística (pone variables en escala comparable)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("\nCómo interpretarlo:")
print("- stratify mantiene el % de ataques en train y test.")
print("- No usamos attack_cat como entrada: eso revelaría la respuesta.")
print("- StandardScaler no cambia el orden de importancia relativa para Random Forest.")

## 3.2 Entrenar modelos (regresión logística + Random Forest)

**Qué hacemos:** entrenamos una regresión logística (modelo lineal interpretable) y un Random Forest para comparar.  
**Por qué:** ver si un modelo más flexible mejora la detección de ataques.

In [ ]:
# Regresión logística: estima la probabilidad de ataque
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)

# Random Forest (opcional de comparación): conjunto de árboles
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=18,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)  # RF no necesita escalado
y_pred_rf = rf.predict(X_test)

print("Modelos entrenados: LogisticRegression y RandomForestClassifier")
print("class_weight='balanced' mitiga el desbalance entre normal y ataque.")

## 3.3 Evaluación: accuracy, precisión, recall y matriz de confusión

**Qué hacemos:** medimos qué tan bien clasifica cada modelo, con énfasis en el recall de ataque.  
**Por qué:** en ciberseguridad, un falso negativo (ataque no detectado) suele ser más grave que una falsa alarma.

In [ ]:
def metricas(nombre, y_true, y_pred):
    return {
        "modelo": nombre,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision_ataque": round(precision_score(y_true, y_pred, pos_label=1), 4),
        "recall_ataque": round(recall_score(y_true, y_pred, pos_label=1), 4),
        "f1_ataque": round(f1_score(y_true, y_pred, pos_label=1), 4),
    }

tabla_metricas = pd.DataFrame([
    metricas("Regresión logística", y_test, y_pred_lr),
    metricas("Random Forest", y_test, y_pred_rf),
])
display(tabla_metricas)

print("\nReporte detallado — Regresión logística:")
print(classification_report(y_test, y_pred_lr, target_names=["Normal", "Ataque"]))

print("Reporte detallado — Random Forest:")
print(classification_report(y_test, y_pred_rf, target_names=["Normal", "Ataque"]))

print("Cómo interpretarlo:")
print("- Accuracy: % de aciertos totales.")
print("- Precisión de ataque: de las alertas, cuántas eran ataques de verdad.")
print("- Recall de ataque: de todos los ataques reales, cuántos detectamos.")
print("- En seguridad priorizamos recall de ataque: fallar un ataque cuesta más que una falsa alarma.")

## 3.4 Matriz de confusión (gráfica)

**Qué hacemos:** visualizamos aciertos y errores de cada modelo.  
**Por qué:** deja claro cuántos ataques se escaparon (falsos negativos).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, titulo in [
    (axes[0], y_pred_lr, "Matriz de confusión — Regresión logística"),
    (axes[1], y_pred_rf, "Matriz de confusión — Random Forest"),
]:
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Ataque"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(titulo)

plt.tight_layout()
plt.savefig("img/matriz_confusion_modelos.png", dpi=150, bbox_inches="tight")
plt.show()

print("Cómo interpretarlo:")
print("- Diagonal = aciertos (normal bien / ataque bien).")
print("- Arriba-derecha: falsa alarma (normal marcado como ataque).")
print("- Abajo-izquierda: falso negativo (ataque marcado como normal) → el más crítico.")

## 3.5 Variables más importantes e interpretación

**Qué hacemos:** vemos qué variables pesan más en cada modelo.  
**Por qué:** explica el resultado en lenguaje de negocio/seguridad, no solo con métricas.

In [ ]:
# Importancia en Random Forest
imp_rf = (
    pd.DataFrame({"variable": X.columns, "importancia_rf": rf.feature_importances_})
    .sort_values("importancia_rf", ascending=False)
    .head(15)
)

# Coeficientes de regresión logística (valor absoluto = fuerza de la señal)
imp_lr = (
    pd.DataFrame({
        "variable": X.columns,
        "coef_abs_lr": np.abs(log_reg.coef_[0]),
        "coef_lr": log_reg.coef_[0],
    })
    .sort_values("coef_abs_lr", ascending=False)
    .head(15)
)

print("Top variables — Random Forest:")
display(imp_rf.reset_index(drop=True))

print("Top variables — Regresión logística (|coeficiente|):")
display(imp_lr.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=imp_rf, y="variable", x="importancia_rf", color="#1565C0", ax=ax)
ax.set_title("Variables más importantes (Random Forest)")
ax.set_xlabel("Importancia")
ax.set_ylabel("Variable")
plt.tight_layout()
plt.savefig("img/importancia_variables_rf.png", dpi=150, bbox_inches="tight")
plt.show()

mejor = tabla_metricas.sort_values("recall_ataque", ascending=False).iloc[0]
print("\nResumen del mejor modelo según recall de ataque:")
print(dict(mejor))

print("\nCómo interpretarlo:")
print("- Variables top (p. ej. sttl, ct_*, rate, sbytes) son señales prácticas para un SOC.")
print("- Si RF supera a logística en recall, patrones no lineales ayudan a detectar ataques.")
print("- El modelo no 'entiende' seguridad: aprende correlaciones históricas del dataset.")

## 3.6 Aplicación profesional y conclusión

**Qué hacemos:** proponemos un uso concreto en un equipo de seguridad y cerramos el proyecto.  
**Por qué:** conecta el ejercicio académico con un escenario laboral real.

In [ ]:
print("Aplicación profesional propuesta")
print("-" * 60)
print("Filtro de alertas para un SOC (Security Operations Center):")
print("1) El modelo puntúa cada flujo de red como probabilidad de ataque.")
print("2) Solo los casos de alta probabilidad llegan a la cola de analistas.")
print("3) Se prioriza recall alto para reducir ataques no vistos.")
print("4) Las falsas alarmas se revisan y se usan para reentrenar el modelo.")
print()
print("Conclusión")
print("-" * 60)
print("Con UNSW-NB15 demostramos que la ciencia de datos puede identificar")
print("patrones asociados a amenazas y anticipar tráfico sospechoso.")
print("La estadística reveló diferencias entre normal y malicioso; el modelo")
print("convirtió esas señales en predicciones evaluables (accuracy y recall).")
print("El siguiente paso profesional sería validar con tráfico propio de la")
print("organización y combinar el modelo con reglas de negocio del equipo de seguridad.")

### Texto para el video — Parte 3

En la Parte 3 definimos la variable objetivo ataque/normal, separamos entrenamiento y prueba, y entrenamos regresión logística y Random Forest. Evaluamos con accuracy, precisión y, sobre todo, recall de ataque, porque dejar pasar una amenaza es más grave que una falsa alarma. La matriz de confusión muestra aciertos y errores, y las variables más importantes confirman señales útiles para un filtro de alertas en un SOC. En conclusión, la ciencia de datos sí ayuda a anticipar amenazas cuando se combina estadística, un buen dataset y métricas alineadas con el riesgo.

---
## Checklist del proyecto

**Parte 1** dataset justificado, documentado, explorado, limpio y CSV guardado  
**Parte 2** media/mediana/moda, dispersión, comparación, outliers, hipótesis + gráficas en `img/`  
**Parte 3** modelo, métricas, matriz de confusión, importancia, aplicación y conclusión  

**Gráficas guardadas:** `img/barras_normal_vs_maliciosa.png`, `img/histogramas_variables_clave.png`, `img/boxplots_outliers.png`, `img/mapa_calor_correlaciones.png`, `img/matriz_confusion_modelos.png`, `img/importancia_variables_rf.png`